# Imports, Device, Global Config & Utility Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import defaultdict, deque, namedtuple

import gymnasium as gym

import torch
import torch.nn as nn
import torch.optim as optim


FAST_MODE = True  # Set to False if you want longer, more accurate runs

if FAST_MODE:
    BJ_EPISODES = 2000      # Blackjack episodes per run
    CP_EPISODES = 500       # CartPole tabular episodes per run
    DQN_EPISODES = 200      # DQN training episodes
    RAINBOW_EPISODES = 200  # Rainbow-lite episodes
else:
    BJ_EPISODES = 10000
    CP_EPISODES = 2000
    DQN_EPISODES = 500
    RAINBOW_EPISODES = 500

SEEDS = [0, 1, 2]  # Seeds for averaging results

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def set_global_seed(seed: int):
    """Set random seeds for numpy, python, and torch."""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Epsilon-Greedy & Dynamic Programming (VI / PI)

In [ ]:
def epsilon_greedy_fast(Q, s_idx, n_actions, eps, rng):
    """Epsilon-greedy policy over a tabular Q-table.

    Q      : np.array [n_states, n_actions]
    s_idx  : current state index (int)
    eps    : exploration probability
    rng    : np.random.Generator
    """
    if rng.random() < eps:
        return int(rng.integers(0, n_actions))
    return int(np.argmax(Q[s_idx]))


def value_iteration_fast(P, R, gamma=1.0, theta=1e-3, max_iter=80):
    """Fast, vectorized value iteration for finite MDP.

    P: [S, A, S'] transition probabilities
    R: [S, A, S'] rewards
    """
    P = np.asarray(P)
    R = np.asarray(R)
    S, A, _ = P.shape

    V = np.zeros(S, dtype=np.float64)
    delta_hist = []

    for _ in range(max_iter):
        # Q(s,a) = sum_s' P(s,a,s') * [R(s,a,s') + gamma * V(s')]
        Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        V_new = Q.max(axis=1)
        delta = np.max(np.abs(V_new - V))
        delta_hist.append(float(delta))
        V = V_new
        if delta < theta:
            break

    policy = Q.argmax(axis=1)
    return V, policy, delta_hist


def policy_evaluation_fast(policy, P, R, gamma=1.0, theta=1e-3, max_iter=50):
    P = np.asarray(P)
    R = np.asarray(R)
    S, A, _ = P.shape
    V = np.zeros(S, dtype=np.float64)

    for _ in range(max_iter):
        Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        V_new = Q[np.arange(S), policy]
        delta = np.max(np.abs(V_new - V))
        V = V_new
        if delta < theta:
            break
    return V


# -------------------------------------------------
# Policy Iteration (returns V, greedy policy and changes per iteration)
# -------------------------------------------------
def policy_iteration_fast_fixed(P, R, gamma=1.0, theta=1e-3, max_iter=40):
    P = np.asarray(P)
    R = np.asarray(R)
    S, A, _ = P.shape

    # Start with a random policy to avoid trivial fixed points
    policy = np.random.randint(0, A, size=S)
    changes_hist = []

    for it in range(max_iter):
        # Policy evaluation
        V = np.zeros(S)
        for _ in range(30):
            Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
            V_new = Q[np.arange(S), policy]
            if np.max(np.abs(V_new - V)) < theta:
                break
            V = V_new

        # Policy improvement
        Q_full = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        new_policy = Q_full.argmax(axis=1)
        changes = int(np.sum(new_policy != policy))
        changes_hist.append(changes)
        policy = new_policy

        # Stop if stable for a few iterations
        if it >= 3 and changes == 0:
            break

    return V, policy, changes_hist


# Blackjack — Model-Based DP (Value Iteration & Policy Iteration)

In [ ]:

# Example usage (uncomment once P_bj, R_bj, bj_gamma are defined):
V_vi_bj, pi_vi_bj, vi_delta_bj = value_iteration_fast(
     P_bj, R_bj, gamma=bj_gamma, theta=1e-2, max_iter=200)
#
# V_pi_bj, pi_pi_bj, pi_changes_bj = policy_iteration_fast_fixed(
#     P_bj, R_bj, gamma=bj_gamma, theta=1e-2, max_iter=50
# )
#
# print("Blackjack VI iterations:", len(vi_delta_bj))
# print("Blackjack PI iterations:", len(pi_changes_bj))
#
# plt.figure()
# plt.plot(vi_delta_bj)
# plt.yscale("log")
# plt.xlabel("Iteration")
# plt.ylabel("Max |ΔV|")
# plt.title("Blackjack Value Iteration Convergence (fast)")
# plt.show()
#
# plt.figure()
# plt.plot(pi_changes_bj)
# plt.xlabel("Policy Iteration")
# plt.ylabel("# state policy changes")
# plt.title("Blackjack Policy Iteration Convergence (fast)")
# plt.show()


## 4️⃣ Blackjack — Model-Free (SARSA vs Q-Learning)

In [ ]:
# -------------------------------------------------
# Compact integer encoding for Blackjack states
# -------------------------------------------------
def encode_bj_state(state):
    """Encode Blackjack state (player_sum, dealer, usable_ace) -> small int index."""
    ps, dealer, ace = state
    ps = max(4, min(ps, 31)) - 4      # 0–27
    dealer = dealer - 1               # 0–9
    ace = 1 if ace else 0             # 0 or 1
    return ps * 20 + dealer * 2 + ace # up to ~1120 states


# -------------------------------------------------
# Fast SARSA for Blackjack
# -------------------------------------------------
def run_sarsa_blackjack_fast(
    num_episodes=BJ_EPISODES,
    alpha=0.1,
    gamma=1.0,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_episodes=None,
    seed=0
):
    env = gym.make("Blackjack-v1", sab=True)
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    n_actions = env.action_space.n
    n_states = 1120  # size of encoded state space
    Q = np.zeros((n_states, n_actions), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / eps_decay_episodes)
        state, _ = env.reset(seed=int(rng.integers(1_000_000)))
        s_idx = encode_bj_state(state)
        action = epsilon_greedy_fast(Q, s_idx, n_actions, eps, rng)

        done = False
        ep_return = 0.0

        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward

            ns_idx = encode_bj_state(next_state)

            if not done:
                next_action = epsilon_greedy_fast(Q, ns_idx, n_actions, eps, rng)
                target = reward + gamma * Q[ns_idx, next_action]
            else:
                next_action = 0
                target = reward

            td_error = target - Q[s_idx, action]
            Q[s_idx, action] += alpha * td_error

            s_idx, action = ns_idx, next_action

        returns[ep] = ep_return

    env.close()
    return Q, returns


# -------------------------------------------------
# Fast Q-Learning for Blackjack
# -------------------------------------------------
def run_q_learning_blackjack_fast(
    num_episodes=BJ_EPISODES,
    alpha=0.1,
    gamma=1.0,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_episodes=None,
    seed=0
):
    env = gym.make("Blackjack-v1", sab=True)
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    n_actions = env.action_space.n
    n_states = 1120
    Q = np.zeros((n_states, n_actions), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / eps_decay_episodes)
        state, _ = env.reset(seed=int(rng.integers(1_000_000)))
        s_idx = encode_bj_state(state)
        done = False
        ep_return = 0.0

        while not done:
            action = epsilon_greedy_fast(Q, s_idx, n_actions, eps, rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward

            ns_idx = encode_bj_state(next_state)

            if not done:
                target = reward + gamma * np.max(Q[ns_idx])
            else:
                target = reward

            td_error = target - Q[s_idx, action]
            Q[s_idx, action] += alpha * td_error

            s_idx = ns_idx

        returns[ep] = ep_return

    env.close()
    return Q, returns


# -------------------------------------------------
# Aggregate Blackjack results over multiple seeds
# -------------------------------------------------
def aggregate_blackjack_fast(algo_fn, seeds, **kwargs):
    all_returns = []
    for seed in seeds:
        _, ret = algo_fn(seed=seed, **kwargs)
        all_returns.append(ret)
    arr = np.stack(all_returns)
    return arr.mean(axis=0), arr.std(axis=0)


# Example usage (uncomment to run):
# sarsa_mean_bj, sarsa_std_bj = aggregate_blackjack_fast(
#     run_sarsa_blackjack_fast,
#     SEEDS,
#     num_episodes=BJ_EPISODES
# )
# ql_mean_bj, ql_std_bj = aggregate_blackjack_fast(
#     run_q_learning_blackjack_fast,
#     SEEDS,
#     num_episodes=BJ_EPISODES
# )
#
# episodes_bj = np.arange(BJ_EPISODES)
# plt.figure()
# plt.plot(episodes_bj, sarsa_mean_bj, label="SARSA")
# plt.fill_between(episodes_bj, sarsa_mean_bj - sarsa_std_bj, sarsa_mean_bj + sarsa_std_bj, alpha=0.3)
# plt.plot(episodes_bj, ql_mean_bj, label="Q-Learning")
# plt.fill_between(episodes_bj, ql_mean_bj - ql_std_bj, ql_mean_bj + ql_std_bj, alpha=0.3)
# plt.xlabel("Episode")
# plt.ylabel("Return")
# plt.title("Blackjack: SARSA vs Q-Learning (mean ± std)")
# plt.legend()
# plt.show()


## 5️⃣ CartPole — Discretization and DP (Value Iteration & Policy Iteration)

In [ ]:
# -------------------------------
# Discretization ranges & bins
# -------------------------------
CART_POS_RANGE = (-2.4, 2.4)
CART_VEL_RANGE = (-3.0, 3.0)
POLE_ANG_RANGE = (-0.209, 0.209)
POLE_ANG_VEL_RANGE = (-3.5, 3.5)

CART_BINS = 3
CART_VEL_BINS = 3
POLE_ANG_BINS = 8
POLE_ANG_VEL_BINS = 12

def create_bins(low, high, num_bins):
    return np.linspace(low, high, num_bins + 1)

cart_pos_bins = create_bins(*CART_POS_RANGE, CART_BINS)
cart_vel_bins = create_bins(*CART_VEL_RANGE, CART_VEL_BINS)
pole_ang_bins = create_bins(*POLE_ANG_RANGE, POLE_ANG_BINS)
pole_ang_vel_bins = create_bins(*POLE_ANG_VEL_RANGE, POLE_ANG_VEL_BINS)


# -------------------------------
# Discretize continuous state
# -------------------------------
def discretize_cartpole_state(state):
    x, x_dot, theta, theta_dot = state

    x = np.clip(x, *CART_POS_RANGE)
    x_dot = np.clip(x_dot, *CART_VEL_RANGE)
    theta = np.clip(theta, *POLE_ANG_RANGE)
    theta_dot = np.clip(theta_dot, *POLE_ANG_VEL_RANGE)

    b0 = np.clip(np.digitize(x, cart_pos_bins) - 1, 0, CART_BINS - 1)
    b1 = np.clip(np.digitize(x_dot, cart_vel_bins) - 1, 0, CART_VEL_BINS - 1)
    b2 = np.clip(np.digitize(theta, pole_ang_bins) - 1, 0, POLE_ANG_BINS - 1)
    b3 = np.clip(np.digitize(theta_dot, pole_ang_vel_bins) - 1, 0, POLE_ANG_VEL_BINS - 1)
    return (b0, b1, b2, b3)


def cartpole_state_index(ds):
    b0, b1, b2, b3 = ds
    return (((b0 * CART_VEL_BINS + b1) * POLE_ANG_BINS + b2) * POLE_ANG_VEL_BINS + b3)


def total_cartpole_states():
    return CART_BINS * CART_VEL_BINS * POLE_ANG_BINS * POLE_ANG_VEL_BINS


N_CP_STATES = total_cartpole_states()
N_CP_ACTIONS = 2

print("CartPole discrete states:", N_CP_STATES, "actions:", N_CP_ACTIONS)


# -------------------------------
# Build a stochastic transition model
# -------------------------------
def bin_center(edges, idx):
    return (edges[idx] + edges[idx + 1]) / 2.0


def rep_state_from_bins(b0, b1, b2, b3):
    return np.array([
        bin_center(cart_pos_bins, b0),
        bin_center(cart_vel_bins, b1),
        bin_center(pole_ang_bins, b2),
        bin_center(pole_ang_vel_bins, b3),
    ], dtype=np.float32)


def build_cartpole_transition_model_stochastic(gamma=0.99, base_jitter=0.01, samples=6):
    """Build a small stochastic MDP model for CartPole using bin centers + jitter.

    This breaks symmetry between actions so policy iteration has non-trivial convergence.
    """
    env = gym.make("CartPole-v1")
    P = np.zeros((N_CP_STATES, N_CP_ACTIONS, N_CP_STATES), dtype=np.float64)
    R = np.zeros((N_CP_STATES, N_CP_ACTIONS, N_CP_STATES), dtype=np.float64)

    for b0 in range(CART_BINS):
        for b1 in range(CART_VEL_BINS):
            for b2 in range(POLE_ANG_BINS):
                for b3 in range(POLE_ANG_VEL_BINS):
                    ds = (b0, b1, b2, b3)
                    s_idx = cartpole_state_index(ds)
                    rep = rep_state_from_bins(b0, b1, b2, b3)

                    for a in range(N_CP_ACTIONS):
                        counts = {}
                        rewards = {}

                        for _ in range(samples):
                            state = rep.copy()

                            # Action-dependent velocity bias + small noise
                            if a == 0:
                                state[1] -= base_jitter
                            else:
                                state[1] += base_jitter
                            state += np.random.uniform(-base_jitter/2, base_jitter/2, size=4)

                            env.reset()
                            env.unwrapped.state = state

                            next_state, reward, terminated, truncated, _ = env.step(a)
                            done = terminated or truncated

                            if done:
                                ns_idx = s_idx
                            else:
                                ds_next = discretize_cartpole_state(next_state)
                                ns_idx = cartpole_state_index(ds_next)

                            counts[ns_idx] = counts.get(ns_idx, 0) + 1
                            rewards[ns_idx] = rewards.get(ns_idx, 0.0) + reward

                        total = sum(counts.values())
                        for ns_idx, c in counts.items():
                            P[s_idx, a, ns_idx] = c / total
                            R[s_idx, a, ns_idx] = rewards[ns_idx] / c

    env.close()
    return P, R


# Build model and run DP
cp_gamma = 0.99
P_cp, R_cp = build_cartpole_transition_model_stochastic()
print("P_cp shape:", P_cp.shape)

V_vi_cp, pi_vi_cp, vi_delta_cp = value_iteration_fast(
    P_cp, R_cp, gamma=cp_gamma, theta=1e-3, max_iter=80
)
V_pi_cp, pi_pi_cp, pi_changes_cp = policy_iteration_fast_fixed(
    P_cp, R_cp, gamma=cp_gamma, theta=1e-3, max_iter=40
)

print("CartPole VI iterations:", len(vi_delta_cp))
print("CartPole PI iterations:", len(pi_changes_cp))

# Plot convergence
plt.figure()
plt.plot(vi_delta_cp)
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Max |ΔV|")
plt.title("CartPole Value Iteration Convergence (fast)")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(pi_changes_cp)
plt.xlabel("Policy Iteration")
plt.ylabel("# state policy changes")
plt.title("CartPole Policy Iteration Convergence (fast, stochastic MDP)")
plt.grid(True)
plt.show()


## 6️⃣ CartPole — Tabular SARSA vs Q-Learning (with discretization)

In [ ]:
def run_sarsa_cartpole(
    num_episodes=CP_EPISODES,
    alpha=0.5,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.01,
    eps_decay_episodes=None,
    seed=0
):
    """Tabular SARSA on discretized CartPole."""
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    Q = np.zeros((N_CP_STATES, N_CP_ACTIONS), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / max(1, eps_decay_episodes))
        obs, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ds = discretize_cartpole_state(obs)
        s_idx = cartpole_state_index(ds)
        action = epsilon_greedy_fast(Q, s_idx, N_CP_ACTIONS, eps, rng)
        done = False
        ep_return = 0.0

        while not done:
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward

            if not done:
                ds_next = discretize_cartpole_state(next_obs)
                ns_idx = cartpole_state_index(ds_next)
                next_action = epsilon_greedy_fast(Q, ns_idx, N_CP_ACTIONS, eps, rng)
                td_target = reward + gamma * Q[ns_idx, next_action]
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error
                s_idx, action = ns_idx, next_action
            else:
                td_target = reward
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error

        returns[ep] = ep_return

    env.close()
    return Q, returns


def run_q_learning_cartpole(
    num_episodes=CP_EPISODES,
    alpha=0.5,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.01,
    eps_decay_episodes=None,
    seed=0
):
    """Tabular Q-Learning on discretized CartPole."""
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    Q = np.zeros((N_CP_STATES, N_CP_ACTIONS), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / max(1, eps_decay_episodes))
        obs, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ds = discretize_cartpole_state(obs)
        s_idx = cartpole_state_index(ds)
        done = False
        ep_return = 0.0

        while not done:
            action = epsilon_greedy_fast(Q, s_idx, N_CP_ACTIONS, eps, rng)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward

            if not done:
                ds_next = discretize_cartpole_state(next_obs)
                ns_idx = cartpole_state_index(ds_next)
                best_next = np.max(Q[ns_idx])
                td_target = reward + gamma * best_next
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error
                s_idx = ns_idx
            else:
                td_target = reward
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error

        returns[ep] = ep_return

    env.close()
    return Q, returns


def aggregate_cartpole(algo_fn, seeds, **kwargs):
    """Aggregate CartPole returns over multiple seeds."""
    all_returns = []
    for seed in seeds:
        _, ret = algo_fn(seed=seed, **kwargs)
        all_returns.append(ret)
    arr = np.stack(all_returns)
    return arr.mean(axis=0), arr.std(axis=0)


# Example usage (uncomment to run):
# sarsa_mean_cp, sarsa_std_cp = aggregate_cartpole(
#     run_sarsa_cartpole,
#     SEEDS,
#     num_episodes=CP_EPISODES,
#     alpha=0.5,
#     gamma=0.99
# )
# ql_mean_cp, ql_std_cp = aggregate_cartpole(
#     run_q_learning_cartpole,
#     SEEDS,
#     num_episodes=CP_EPISODES,
#     alpha=0.5,
#     gamma=0.99
# )
#
# episodes_cp = np.arange(CP_EPISODES)
# plt.figure()
# plt.plot(episodes_cp, sarsa_mean_cp, label="SARSA")
# plt.fill_between(episodes_cp, sarsa_mean_cp - sarsa_std_cp, sarsa_mean_cp + sarsa_std_cp, alpha=0.3)
# plt.plot(episodes_cp, ql_mean_cp, label="Q-Learning")
# plt.fill_between(episodes_cp, ql_mean_cp - ql_std_cp, ql_mean_cp + ql_std_cp, alpha=0.3)
# plt.xlabel("Episode")
# plt.ylabel("Episode length (return)")
# plt.title("CartPole: SARSA vs Q-Learning (mean ± std over seeds)")
# plt.legend()
# plt.show()


## 7️⃣ Extra Credit — DQN & Rainbow-lite for CartPole

In [ ]:
# =======================================================
# Replay Buffers
# =======================================================
Transition = namedtuple("Transition", ("state", "action", "reward", "next_state", "done"))

class ReplayBuffer:
    """Standard experience replay buffer."""
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)


class PrioritizedReplayBuffer:
    """Simplified prioritized experience replay."""
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.eps = 1e-5

    def push(self, *args):
        max_prio = self.priorities.max() if self.buffer else 1.0
        if len(self.buffer) < self.capacity:
            self.buffer.append(Transition(*args))
        else:
            self.buffer[self.pos] = Transition(*args)
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
        probs = prios ** self.alpha
        probs /= probs.sum()
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights /= weights.max()
        weights = torch.tensor(weights, dtype=torch.float32, device=device)

        batch = Transition(*zip(*samples))
        return batch, indices, weights

    def update_priorities(self, indices, priorities):
        for idx, prio in zip(indices, priorities):
            self.priorities[idx] = float(prio + self.eps)

    def __len__(self):
        return len(self.buffer)


# =======================================================
# Networks: DQN & Dueling DQN
# =======================================================
class DQNNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)


class DuelingDQNNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU()
        )
        self.advantage = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        self.value = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        f = self.feature(x)
        adv = self.advantage(f)
        val = self.value(f)
        q = val + (adv - adv.mean(dim=1, keepdim=True))
        return q


# =======================================================
# Vanilla DQN Agent
# =======================================================
class DQNAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000
    ):
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay_steps = eps_decay_steps

        self.online_net = DQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net = DQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_capacity)

        self.total_steps = 0

    def epsilon(self):
        return max(
            self.eps_end,
            self.eps_start - (self.eps_start - self.eps_end) * (self.total_steps / self.eps_decay_steps)
        )

    def select_action(self, state, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        if rng.random() < self.epsilon():
            return int(rng.integers(0, self.action_dim))
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_vals = self.online_net(state_t)
        return int(q_vals.argmax(dim=1).item())

    def push(self, *args):
        self.buffer.push(*args)

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None
        transitions = self.buffer.sample(self.batch_size)
        states = torch.tensor(np.array(transitions.state), dtype=torch.float32, device=device)
        actions = torch.tensor(transitions.action, dtype=torch.int64, device=device).unsqueeze(-1)
        rewards = torch.tensor(transitions.reward, dtype=torch.float32, device=device).unsqueeze(-1)
        next_states = torch.tensor(np.array(transitions.next_state), dtype=torch.float32, device=device)
        dones = torch.tensor(transitions.done, dtype=torch.float32, device=device).unsqueeze(-1)

        q_vals = self.online_net(states).gather(1, actions)
        with torch.no_grad():
            max_next = self.target_net(next_states).max(dim=1, keepdim=True)[0]
            target = rewards + self.gamma * (1.0 - dones) * max_next

        loss = nn.MSELoss()(q_vals, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.total_steps % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        return loss.item()


def train_dqn_cartpole(
    num_episodes=DQN_EPISODES,
    max_steps_per_ep=500,
    seed=0
):
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = DQNAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000
    )

    episode_returns = []
    global_step = 0

    for ep in range(num_episodes):
        state, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ep_return = 0.0

        for t in range(max_steps_per_ep):
            agent.total_steps = global_step
            action = agent.select_action(state, rng=rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.push(state, action, reward, next_state, float(done))
            agent.update()

            state = next_state
            ep_return += reward
            global_step += 1

            if done:
                break

        episode_returns.append(ep_return)
        if (ep + 1) % 50 == 0:
            print(f"[DQN] Episode {ep+1}/{num_episodes}, return={ep_return:.1f}")

    env.close()
    return episode_returns


# =======================================================
# Rainbow-lite Agent (Dueling DQN + PER + Double DQN)
# =======================================================
class RainbowLiteAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        per_beta_steps=50_000
    ):
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update

        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay_steps = eps_decay_steps

        self.per_alpha = per_alpha
        self.per_beta_start = per_beta_start
        self.per_beta_end = per_beta_end
        self.per_beta_steps = per_beta_steps

        self.online_net = DuelingDQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net = DuelingDQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer = PrioritizedReplayBuffer(buffer_capacity, alpha=per_alpha)

        self.total_steps = 0

    def epsilon(self):
        return max(
            self.eps_end,
            self.eps_start - (self.eps_start - self.eps_end) * (self.total_steps / self.eps_decay_steps)
        )

    def beta(self):
        return min(
            self.per_beta_end,
            self.per_beta_start + (self.per_beta_end - self.per_beta_start) * (self.total_steps / self.per_beta_steps)
        )

    def select_action(self, state, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        if rng.random() < self.epsilon():
            return int(rng.integers(0, self.action_dim))
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_vals = self.online_net(state_t)
        return int(q_vals.argmax(dim=1).item())

    def push(self, *args):
        self.buffer.push(*args)

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None
        beta = self.beta()
        transitions, indices, weights = self.buffer.sample(self.batch_size, beta=beta)

        states = torch.tensor(np.array(transitions.state), dtype=torch.float32, device=device)
        actions = torch.tensor(transitions.action, dtype=torch.int64, device=device).unsqueeze(-1)
        rewards = torch.tensor(transitions.reward, dtype=torch.float32, device=device).unsqueeze(-1)
        next_states = torch.tensor(np.array(transitions.next_state), dtype=torch.float32, device=device)
        dones = torch.tensor(transitions.done, dtype=torch.float32, device=device).unsqueeze(-1)

        q_vals = self.online_net(states).gather(1, actions)

        with torch.no_grad():
            next_online = self.online_net(next_states)
            best_actions = next_online.argmax(dim=1, keepdim=True)
            next_target = self.target_net(next_states)
            next_q = next_target.gather(1, best_actions)
            target = rewards + self.gamma * (1.0 - dones) * next_q

        td_errors = target - q_vals
        loss = (weights.unsqueeze(-1) * td_errors.pow(2)).mean()

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        new_prios = td_errors.detach().abs().squeeze(-1).cpu().numpy()
        self.buffer.update_priorities(indices, new_prios)

        if self.total_steps % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        return loss.item()


def train_rainbow_lite_cartpole(
    num_episodes=RAINBOW_EPISODES,
    max_steps_per_ep=500,
    seed=0
):
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = RainbowLiteAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        per_beta_steps=50_000
    )

    episode_returns = []
    global_step = 0

    for ep in range(num_episodes):
        state, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ep_return = 0.0

        for t in range(max_steps_per_ep):
            agent.total_steps = global_step
            action = agent.select_action(state, rng=rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.push(state, action, reward, next_state, float(done))
            agent.update()

            state = next_state
            ep_return += reward
            global_step += 1

            if done:
                break

        episode_returns.append(ep_return)
        if (ep + 1) % 50 == 0:
            print(f"[Rainbow-lite] Episode {ep+1}/{num_episodes}, return={ep_return:.1f}")

    env.close()
    return episode_returns


# Example usage (uncomment to run; can be slow if episodes are large):
# dqn_returns = train_dqn_cartpole(num_episodes=DQN_EPISODES, max_steps_per_ep=500, seed=0)
# rainbow_returns = train_rainbow_lite_cartpole(num_episodes=RAINBOW_EPISODES, max_steps_per_ep=500, seed=0)
# episodes = np.arange(len(dqn_returns))
# plt.figure()
# plt.plot(episodes, dqn_returns, label="DQN")
# plt.plot(episodes, rainbow_returns, label="Rainbow-lite")
# plt.xlabel("Episode")
# plt.ylabel("Return (episode length)")
# plt.title("CartPole: DQN vs Rainbow-lite (extra credit)")
# plt.legend()
# plt.show()
